# NER with BERT + CRF


In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers pytorch-crf seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
# Cell 2: Imports
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torchcrf import CRF
from seqeval.metrics import classification_report, precision_score, recall_score, f1_score
import numpy as np
from collections import Counter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Using device: cuda
GPU: Tesla T4


In [ ]:
# Cell 3: Config
MODEL_NAME = 'bert-base-cased'
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 10
LR = 3e-5
DROPOUT = 0.1

TRAIN_FILE = 'train.conll'
VALID_FILE = 'valid.conll'
TEST_FILE  = 'test.conll'

In [ ]:
# Cell 4: Read CoNLL files
def read_conll(filepath):
    """Read CoNLL file, return list of (tokens, tags) per document."""
    sentences, tokens, tags = [], [], []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                if tokens:
                    sentences.append((tokens, tags))
                    tokens, tags = [], []
            else:
                parts = line.split('\t')
                if len(parts) == 2:
                    tokens.append(parts[0])
                    tags.append(parts[1])
    if tokens:
        sentences.append((tokens, tags))
    return sentences

train_data = read_conll(TRAIN_FILE)
valid_data = read_conll(VALID_FILE)
test_data  = read_conll(TEST_FILE)

print(f'Train: {len(train_data)} docs')
print(f'Valid: {len(valid_data)} docs')
print(f'Test:  {len(test_data)} docs')

Train: 2527 docs
Valid: 316 docs
Test:  316 docs


In [ ]:
# Cell 5: Build label mapping
all_tags = set()
for tokens, tags in train_data + valid_data + test_data:
    all_tags.update(tags)

label_list = sorted(all_tags)
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
num_labels = len(label_list)

print(f'Labels ({num_labels}): {label_list}')

Labels (23): ['B-BUY_G', 'B-BUY_N', 'B-GSTL', 'B-GT_AMT', 'B-GT_AMTL', 'B-INV_DL', 'B-INV_DT', 'B-INV_L', 'B-INV_NO', 'B-SUPP_G', 'B-SUPP_N', 'I-BUY_G', 'I-BUY_N', 'I-GSTL', 'I-GT_AMT', 'I-GT_AMTL', 'I-INV_DL', 'I-INV_DT', 'I-INV_L', 'I-INV_NO', 'I-SUPP_G', 'I-SUPP_N', 'O']


In [ ]:
# Cell 6: Dataset class
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class NERDataset(Dataset):
    def __init__(self, data, tokenizer, label2id, max_len):
        self.data = data
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens, tags = self.data[idx]
        # Truncate to max_len - 2 (for [CLS] and [SEP])
        tokens = tokens[:self.max_len - 2]
        tags = tags[:self.max_len - 2]

        # Tokenize word by word, align labels to subwords
        input_ids = [self.tokenizer.cls_token_id]
        label_ids = [self.label2id['O']]  # CLS gets O

        for word, tag in zip(tokens, tags):
            word_tokens = self.tokenizer.encode(word, add_special_tokens=False)
            if not word_tokens:
                continue
            # Only first subword gets the real label, rest get same label
            input_ids.extend(word_tokens)
            label_ids.append(self.label2id[tag])
            # For extra subword tokens, use same tag (I- if B-, else same)
            for _ in range(1, len(word_tokens)):
                if tag.startswith('B-'):
                    label_ids.append(self.label2id['I-' + tag[2:]])
                else:
                    label_ids.append(self.label2id[tag])

        # Truncate if subword expansion exceeded max_len
        input_ids = input_ids[:self.max_len - 1]
        label_ids = label_ids[:self.max_len - 1]

        # Add [SEP]
        input_ids.append(self.tokenizer.sep_token_id)
        label_ids.append(self.label2id['O'])

        # Padding
        seq_len = len(input_ids)
        pad_len = self.max_len - seq_len
        attention_mask = [1] * seq_len + [0] * pad_len
        input_ids += [self.tokenizer.pad_token_id] * pad_len
        label_ids += [self.label2id['O']] * pad_len

        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'labels': torch.tensor(label_ids, dtype=torch.long),
            'seq_len': seq_len
        }

train_ds = NERDataset(train_data, tokenizer, label2id, MAX_LEN)
valid_ds = NERDataset(valid_data, tokenizer, label2id, MAX_LEN)
test_ds  = NERDataset(test_data,  tokenizer, label2id, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)

print(f'Train batches: {len(train_loader)}, Valid: {len(valid_loader)}, Test: {len(test_loader)}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Train batches: 158, Valid: 20, Test: 20


In [ ]:
# Cell 7: BERT + CRF Model
class BERT_CRF(nn.Module):
    def __init__(self, model_name, num_labels, dropout=0.1):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)
        self.crf = CRF(num_tags=num_labels, batch_first=True)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        emissions = self.classifier(self.dropout(outputs.last_hidden_state))
        mask = attention_mask.bool()

        if labels is not None:
            loss = -self.crf(emissions, labels, mask=mask, reduction='mean')
            return loss
        else:
            return self.crf.decode(emissions, mask=mask)

model = BERT_CRF(MODEL_NAME, num_labels, DROPOUT).to(device)
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model parameters: 108,328,534


In [ ]:
# Cell 8: Optimizer & Scheduler
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
)

In [ ]:
# Cell 9: Evaluation function
def evaluate(model, dataloader, id2label):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels']
            seq_lens = batch['seq_len']

            preds = model(input_ids, attention_mask)

            for i in range(len(preds)):
                slen = seq_lens[i].item()
                # Skip [CLS] and [SEP]: indices 1 to slen-2
                true_tags = [id2label[labels[i][j].item()] for j in range(1, slen - 1)]
                pred_tags = [id2label[preds[i][j]] for j in range(1, slen - 1)]
                y_true.append(true_tags)
                y_pred.append(pred_tags)

    p = precision_score(y_true, y_pred)
    r = recall_score(y_true, y_pred)
    f = f1_score(y_true, y_pred)
    return p, r, f, y_true, y_pred

In [ ]:
# Cell 10: Training loop
best_f1 = 0.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0

    for step, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        loss = model(input_ids, attention_mask, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        total_loss += loss.item()

        if (step + 1) % 50 == 0:
            print(f'  Epoch {epoch} Step {step+1}/{len(train_loader)} Loss: {loss.item():.4f}')

    avg_loss = total_loss / len(train_loader)
    p, r, f1, _, _ = evaluate(model, valid_loader, id2label)
    print(f'Epoch {epoch}/{EPOCHS} - Loss: {avg_loss:.4f} | Val P: {p:.4f} R: {r:.4f} F1: {f1:.4f}')

    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), 'best_bert_crf.pt')
        print(f'  -> Saved best model (F1: {f1:.4f})')

print(f'\nTraining complete. Best Val F1: {best_f1:.4f}')

  Epoch 1 Step 50/158 Loss: 58.6933
  Epoch 1 Step 100/158 Loss: 9.2907
  Epoch 1 Step 150/158 Loss: 1.3119
Epoch 1/10 - Loss: 66.9268 | Val P: 0.9368 R: 0.9681 F1: 0.9522
  -> Saved best model (F1: 0.9522)
  Epoch 2 Step 50/158 Loss: 1.5011
  Epoch 2 Step 100/158 Loss: 1.8004
  Epoch 2 Step 150/158 Loss: 0.4351
Epoch 2/10 - Loss: 1.6544 | Val P: 0.9875 R: 0.9889 F1: 0.9882
  -> Saved best model (F1: 0.9882)
  Epoch 3 Step 50/158 Loss: 3.8658
  Epoch 3 Step 100/158 Loss: 0.0650
  Epoch 3 Step 150/158 Loss: 0.0195
Epoch 3/10 - Loss: 1.1829 | Val P: 0.9903 R: 0.9889 F1: 0.9896
  -> Saved best model (F1: 0.9896)
  Epoch 4 Step 50/158 Loss: 0.2464
  Epoch 4 Step 100/158 Loss: 0.0134
  Epoch 4 Step 150/158 Loss: 3.8519
Epoch 4/10 - Loss: 0.8708 | Val P: 0.9944 R: 0.9875 F1: 0.9909
  -> Saved best model (F1: 0.9909)
  Epoch 5 Step 50/158 Loss: 0.0191
  Epoch 5 Step 100/158 Loss: 0.4555
  Epoch 5 Step 150/158 Loss: 0.0139
Epoch 5/10 - Loss: 0.6815 | Val P: 0.9889 R: 0.9917 F1: 0.9903
  Epoch 

In [ ]:
# Cell 11: Test evaluation
model.load_state_dict(torch.load('best_bert_crf.pt'))
p, r, f1, y_true, y_pred = evaluate(model, test_loader, id2label)

print('='*60)
print('TEST RESULTS - BERT + CRF')
print('='*60)
print(f'Precision: {p:.4f}')
print(f'Recall:    {r:.4f}')
print(f'F1-Score:  {f1:.4f}')
print('='*60)
print()
print(classification_report(y_true, y_pred))

TEST RESULTS - BERT + CRF
Precision: 0.9837
Recall:    0.9932
F1-Score:  0.9884

              precision    recall  f1-score   support

       BUY_G       0.98      0.98      0.98        62
       BUY_N       0.94      0.99      0.96        68
        GSTL       0.99      1.00      1.00       119
      GT_AMT       1.00      1.00      1.00        22
     GT_AMTL       1.00      1.00      1.00        23
      INV_DL       0.99      0.97      0.98        75
      INV_DT       0.99      0.99      0.99        74
       INV_L       0.99      1.00      0.99        75
      INV_NO       0.97      1.00      0.99        75
      SUPP_G       0.98      1.00      0.99        57
      SUPP_N       1.00      1.00      1.00        81

   micro avg       0.98      0.99      0.99       731
   macro avg       0.99      0.99      0.99       731
weighted avg       0.98      0.99      0.99       731



In [ ]:
# Cell 12 (Optional): Save model for deployment
import json

os.makedirs('saved_model', exist_ok=True)
torch.save(model.state_dict(), 'saved_model/model.pt')
with open('saved_model/labels.json', 'w') as f:
    json.dump({'label2id': label2id, 'id2label': {str(k): v for k, v in id2label.items()}}, f)
tokenizer.save_pretrained('saved_model')
print('Model saved to saved_model/')

Model saved to saved_model/
